# COMP8210 Assignment 2 – Knowledge Lake data preparation
**Dataset:** BBC News (Greene & Cunningham, ICML 2006), 2,225 articles, 5 categories.

**Before you start:** upload `bbc-fulltext.zip` into a Google Drive folder called **`COMP8210`** (inside *My Drive*).

**How to run:** put your email in Step 1, then use **Runtime → Run all**. Each step saves its result to your Drive folder.
Screenshot the outputs marked 📸 – they are evidence for your report.

## Step 1 – Settings and Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)   # hides harmless pandas notices

DIR = '/content/drive/MyDrive/COMP8210/'        # your Drive folder
os.makedirs(DIR, exist_ok=True)

# Wikimedia asks scripts to include a contact in their User-Agent (used only in Step 8)
EMAIL = 'YOUR_EMAIL_HERE'

## Step 2 – Install spaCy (the NER model)

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

## Step 3 – Load all 2,225 articles 📸

In [ ]:
import glob
import pandas as pd

ZIP = DIR + 'bbc-fulltext.zip'
assert os.path.exists(ZIP), 'Upload bbc-fulltext.zip to My Drive/COMP8210 first'
!unzip -q -o "{ZIP}" -d /content/

rows = []
for path in sorted(glob.glob('/content/bbc/*/*.txt')):
    with open(path, encoding='latin-1') as f:
        lines = f.read().strip().split('\n')
    rows.append({'category': path.split('/')[-2],
                 'file': os.path.basename(path),
                 'title': lines[0].strip(),
                 'text': '\n'.join(lines[1:]).strip()})

df = pd.DataFrame(rows)
df.to_csv(DIR + 'bbc_full.csv', index=False)
print(len(df), 'articles')                 # should be 2225
print(df['category'].value_counts())

## Step 4 – Stratified sample: 40 articles per category (200 total)

In [ ]:
sample = pd.concat([g.sample(40, random_state=42)
                    for _, g in df.groupby('category')]).reset_index(drop=True)
sample['doc_id'] = sample.index
sample.to_csv(DIR + 'bbc_sample.csv', index=False)
print(sample['category'].value_counts())

## Step 5 – Extract entities with spaCy (PERSON, ORG, GPE) 📸

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

rows = []
for doc_id, cat, title, text in zip(sample.doc_id, sample.category, sample.title, sample.text):
    for ent in nlp(title + '. ' + text).ents:
        if ent.label_ in {'PERSON', 'ORG', 'GPE'}:
            rows.append({'doc_id': doc_id, 'category': cat,
                         'entity': ent.text.strip(), 'type': ent.label_})

ents = pd.DataFrame(rows).drop_duplicates()
ents.to_csv(DIR + 'entities.csv', index=False)          # RAW entities (the 'before cleaning' version)

print(len(ents), 'entity mentions')
print(ents['type'].value_counts())
print(ents['entity'].value_counts().head(20))

## Step 6 – Check inconsistent entity types 📸
The same name gets different labels in different articles – a weakness of small statistical NER models (Section 1.1a).

In [ ]:
check = ['Oscar', 'Chelsea', 'Clarke', 'Blair', 'Labour']
print(ents[ents.entity.isin(check)].groupby(['entity', 'type']).size())

## Step 7 – Clean entities and build co-occurrence relationships 📸

In [ ]:
from itertools import combinations

ALIASES = {'Britain': 'UK', 'United Kingdom': 'UK', 'U.S.': 'US',
           'United States': 'US', 'America': 'US',
           'Lib Dem': 'Lib Dems', 'Blair': 'Tony Blair'}   # assumption: 'Blair' = Tony Blair in 2004-05 news
DROP = {'Parliament', 'Digital', 'GMT'}                      # too generic / not a real entity

def clean_doc(g):
    g = g.copy()
    g['entity'] = (g['entity']
                   .str.replace(r'^[Tt]he ', '', regex=True)                         # 'the BBC' -> 'BBC'
                   .str.replace(r'^(Mr|Mrs|Ms|Miss|Dr|Sir)\.? ', '', regex=True)     # 'Mr Blair' -> 'Blair'
                   .str.replace(r"['’]s$", '', regex=True)                      # "BBC Radio 4's" -> 'BBC Radio 4'
                   .str.strip()
                   .replace(ALIASES))
    full_names = [e for e in g.loc[g.type == 'PERSON', 'entity'] if ' ' in e]
    def expand(e):                 # 'Howard' -> 'Michael Howard' if the full name is in the same article
        if ' ' not in e:
            m = [p for p in full_names if p.split()[-1] == e]
            if len(m) == 1:
                return m[0]
        return e
    g['entity'] = g['entity'].map(expand)
    return g

clean = (pd.concat([clean_doc(g) for _, g in ents.groupby('doc_id')])
             .query('entity not in @DROP')
             .drop_duplicates(['doc_id', 'entity']))

# one type per entity = the label spaCy used most often
main_type = clean.groupby('entity')['type'].agg(lambda s: s.mode()[0])
clean['type'] = clean['entity'].map(main_type)
# main category per entity = where it appears most often (used in Step 8)
main_cat = clean.groupby('entity')['category'].agg(lambda s: s.mode()[0])
clean.to_csv(DIR + 'entities_clean.csv', index=False)

# relationship = two entities mentioned in the same article; weight = number of shared articles
pairs = [(a, b, d) for d, g in clean.groupby('doc_id')
         for a, b in combinations(sorted(set(g.entity)), 2)]
rel = pd.DataFrame(pairs, columns=['source', 'target', 'doc_id'])
edges = rel.groupby(['source', 'target']).size().reset_index(name='weight')
edges.to_csv(DIR + 'relations.csv', index=False)

print('before:', ents.entity.nunique(), 'unique entities | after:', clean.entity.nunique())
print(len(edges), 'relationships;', (edges.weight > 1).sum(), 'appear in 2+ articles')
print(clean['entity'].value_counts().head(15))
print(edges.sort_values('weight', ascending=False).head(10))

## Step 8 – Link the top 50 entities to Wikidata 📸
Two methods are compared:
* **naive** – take Wikidata's first search result
* **context-aware** – take the top 10 results, drop names/languages, and prefer the one whose description fits the entity type and the topic of its articles

Takes about 1–2 minutes.

In [ ]:
import requests, time

session = requests.Session()
session.headers.update({'User-Agent': f'COMP8210-KnowledgeLake/1.0 (Macquarie University student project; {EMAIL})'})
API = 'https://www.wikidata.org/w/api.php'

def candidates(name, limit=10, tries=4):
    for attempt in range(tries):
        r = session.get(API, params={'action': 'wbsearchentities', 'search': name, 'language': 'en',
                                     'format': 'json', 'limit': limit, 'maxlag': 5}, timeout=20)
        if r.status_code == 200 and r.headers.get('content-type', '').startswith('application/json'):
            d = r.json()
            if 'error' in d:              # server busy: wait and retry
                time.sleep(5)
                continue
            return d.get('search', [])
        print(f'  {name}: HTTP {r.status_code}')
        time.sleep(2 ** attempt * 2)      # back off: 2s, 4s, 8s, 16s
    return []

ABBREV = {'UK': 'United Kingdom', 'EU': 'European Union', 'US': 'United States of America'}
BAD = ['given name', 'family name', 'surname', 'language', 'disambiguation']
CAT_WORDS = {
    'politics': ['politic', 'party', 'minister', 'government', 'parliament', 'secretary', 'chancellor', 'british', 'united kingdom'],
    'business': ['company', 'corporation', 'bank', 'business', 'conglomerate', 'telecom', 'multinational'],
    'tech': ['technology', 'software', 'company', 'computer', 'internet', 'video game', 'telecom'],
    'sport': ['football', 'club', 'rugby', 'tennis', 'athlet', 'cricket', 'player', 'team', 'sport'],
    'entertainment': ['film', 'actor', 'actress', 'singer', 'band', 'music', 'award', 'television', 'director']}
TYPE_WORDS = {
    'PERSON': ['politician', 'player', 'actor', 'actress', 'singer', 'businessman', 'athlete', 'director', 'musician', 'minister'],
    'GPE': ['country', 'city', 'state', 'capital', 'region', 'town', 'island'],
    'ORG': ['company', 'party', 'club', 'organisation', 'organization', 'broadcaster', 'agency', 'department', 'band', 'corporation']}

def score(c, rank, etype, cat):
    desc = (c.get('description') or '').lower()
    if any(b in desc for b in BAD):
        return -99                                                   # names, languages, disambiguation pages
    s = -rank                                                        # trust Wikidata's ranking a bit
    s += min(2, sum(w in desc for w in CAT_WORDS.get(cat, [])))      # fits the articles' topic
    s += min(1, sum(w in desc for w in TYPE_WORDS.get(etype, [])))   # fits the entity type
    return s

top = clean['entity'].value_counts().head(50)
rows = []
for e, n in top.items():
    cands = candidates(ABBREV.get(e, e))
    time.sleep(0.5)
    naive_c = candidates(e) if e in ABBREV else cands               # baseline = raw name, first hit
    naive = naive_c[0] if naive_c else {}
    best = (max(enumerate(cands), key=lambda rc: score(rc[1], rc[0], main_type[e], main_cat[e]))[1]
            if cands else {})
    rows.append((e, main_type[e], main_cat[e], n,
                 naive.get('id'), naive.get('description', ''),
                 best.get('id'), best.get('description', '')))

enriched = pd.DataFrame(rows, columns=['entity', 'type', 'category', 'articles',
                                       'naive_id', 'naive_desc', 'wikidata_id', 'description'])
enriched['changed'] = enriched.naive_id != enriched.wikidata_id
enriched.to_csv(DIR + 'enriched.csv', index=False)

pd.set_option('display.max_colwidth', 55)
print(enriched.changed.sum(), 'links changed by context-aware linking\n')
print(enriched.loc[enriched.changed, ['entity', 'category', 'naive_desc', 'description']].to_string())

## Step 9 – Review all 50 links
Read every row. Wrong matches go into the lists in Step 10.

In [ ]:
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.max_rows', 60)
print(enriched[['entity', 'type', 'category', 'naive_desc', 'description']].to_string())

## Step 10 – Linking accuracy: naive vs context-aware 📸
Edit the two lists if you found other wrong links in Step 9, then run.

In [ ]:
# entities whose Wikidata link is WRONG (add any others you spot in Step 9)
WRONG_NAIVE = ['UK', 'EU', 'Oscar', 'Gordon Brown', 'Charles Clarke', 'BT', 'Chelsea',
               'Liverpool', 'House of Lords', 'European Indoor Championships']
WRONG_NEW   = ['Oscar', 'BT', 'Chelsea', 'European Indoor Championships']

present = set(enriched.entity)
wn = [e for e in WRONG_NAIVE if e in present]
wc = [e for e in WRONG_NEW if e in present]
n = len(enriched)
acc_naive, acc_new = (n - len(wn)) / n, (n - len(wc)) / n
print(f'naive linking:  {n-len(wn)}/{n} ({acc_naive:.0%}) correct')
print(f'context-aware:  {n-len(wc)}/{n} ({acc_new:.0%}) correct')

enriched['correct_naive'] = ~enriched.entity.isin(wn)
enriched['correct_new'] = ~enriched.entity.isin(wc)
enriched.to_csv(DIR + 'enriched.csv', index=False)

## Step 11 – Export files for Neo4j
Only correct Wikidata links go into the graph.

In [ ]:
articles = sample[['doc_id', 'category', 'title']]
good = enriched.loc[enriched.correct_new, ['entity', 'wikidata_id', 'description']]
nodes = (clean.groupby('entity')
              .agg(type=('type', 'first'), articles=('doc_id', 'nunique'))
              .reset_index()
              .merge(good, on='entity', how='left'))

articles.to_csv(DIR + 'neo4j_articles.csv', index=False)                    # (:Article) nodes
nodes.to_csv(DIR + 'neo4j_entities.csv', index=False)                       # (:Entity) nodes + Wikidata
clean[['doc_id', 'entity']].to_csv(DIR + 'neo4j_mentions.csv', index=False) # (:Article)-[:MENTIONS]->(:Entity)
edges.to_csv(DIR + 'neo4j_cooccurs.csv', index=False)                       # (:Entity)-[:CO_OCCURS]->(:Entity)

print(len(articles), 'articles |', len(nodes), 'entities |', nodes.wikidata_id.notna().sum(), 'with Wikidata |',
      len(clean), 'mentions |', len(edges), 'co-occurrences')

## Step 12 – Summary numbers for your report 📸
Copy these into the *Dataset* section and Sections 1.1, 1.2 and 2.2.

In [ ]:
per_doc = clean.groupby('doc_id').size()
print('DATASET SUMMARY')
print(f'  Corpus:                     {len(df)} BBC articles, {df.category.nunique()} categories')
print(f'  Stratified sample:          {len(sample)} articles (40 per category)')
print(f'  Raw entity mentions:        {len(ents)}  ->  ' + ', '.join(f'{k} {v}' for k, v in ents.type.value_counts().items()))
print(f'  Unique entities:            {ents.entity.nunique()} raw -> {clean.entity.nunique()} after cleaning '
      f'({1 - clean.entity.nunique() / ents.entity.nunique():.1%} merged/removed)')
print(f'  Entities per article:       mean {per_doc.mean():.1f}, median {per_doc.median():.0f}')
print(f'  Co-occurrence relationships:{len(edges):>6}  ({(edges.weight > 1).sum()} appear in 2+ articles)')
print(f'  Wikidata linking (top 50):  naive {acc_naive:.0%} -> context-aware {acc_new:.0%}')
print('\nFiles in Drive:', sorted(f for f in os.listdir(DIR) if f.endswith('.csv')))